<!-- paper-paired-role-banner -->
> 🟩 **정답 코드 · 오른쪽 창** — 왼쪽과 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 구현 이유와 텐서 shape를 설명해 보세요.

# 14. DQN — Playing Atari with Deep Reinforcement Learning

- **원 논문:** [Playing Atari with Deep Reinforcement Learning](https://arxiv.org/abs/1312.5602)
- **저자 / 발표:** Volodymyr Mnih et al. · arXiv preprint (2013)
- **난이도 / 예상 시간:** 중급 · 약 65분
- **선수 지식:** PyTorch 기본, 강화학습의 state/action/reward, Bellman equation

이 노트북은 논문 결과 수치를 그대로 재현하는 대규모 benchmark가 아니라,
핵심 수식과 구조가 실제로 어떻게 동작하는지 확인하는 **하드웨어 인식 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하며, 작은 NumPy·Python 실험은
전송 비용을 피하기 위해 CPU에서 실행될 수 있습니다.

## 재현 범위와 완료 기준

**이 노트북이 재현하는 것**

외부 Gym 없이 5-state LineWorld에서 replay memory, epsilon-greedy 행동, 고정된 Bellman target과 Q-network MSE update를 구현해 greedy 정책이 오른쪽 보상에 도달하게 한다.

**원 논문과 다른 것**

논문은 Atari 화면 4장을 convolutional Q-network에 넣어 수백만 frame을 학습한다. 여기서는 one-hot 상태 5개와 2층 MLP로 Algorithm 1의 학습 역학만 재현한다.

완료하려면 TODO를 구현하고, 검증 셀의 `assert`를 모두 통과시키고, 마지막 관찰 질문에
자신의 말로 답하세요. 수치가 논문과 다르더라도 핵심 불변식과 비교 방향이 맞으면 성공입니다.

## 논문 ↔ 노트북 지도

        절 번호와 수식 번호는 위 링크의 대표 공개본을 기준으로 합니다. 판본에 따라 페이지는 달라질 수
        있으므로 **절 제목과 수식 번호**를 함께 사용하세요.

        | 원 논문의 위치 | 이 노트북의 대응 실습 | 확인할 증거 |
        |---|---|---|
        | §2 Background, Eq. (1) | `bellman_targets`의 terminal-aware TD target | done 상태에는 bootstrap 값이 더해지지 않는지 검사 |
| §2, Eq. (2)–(3) | `QNetwork`가 선택한 action의 Q와 고정 target 사이 오차를 최소화 | Q-value shape, finite loss와 gradient를 검사 |
| §4 Deep Reinforcement Learning, Algorithm 1 | `ReplayBuffer`와 `epsilon_greedy`의 수집·표본화·탐험 경로 | buffer batch 계약과 학습 뒤 greedy return을 검사 |

        > 읽기 순서: 먼저 해당 절을 훑고 → 코드를 예측해 작성하고 → 출력이 논문의 주장과 왜
        > 연결되는지 한 문장으로 설명합니다.

## 핵심 수식과 구현 설계

$$
y=r+\gamma(1-d)\max_{a'}Q_{\theta^-}(s',a'),\qquad
\mathcal{L}(\theta)=\mathbb{E}[(y-Q_\theta(s,a))^2]
$$

- $\theta$는 online network, $\theta^-$는 고정된 target network, $d$는 terminal flag입니다.
- replay buffer는 상관된 transition을 섞고 target network는 bootstrapping target을 느리게 변화시킵니다.
- transition 저장, batch sampling, TD target, optimization, target sync를 명시적으로 나눕니다.
- state batch $[B,D_s]$에서 action value $[B,A]$를 만들고 gather로 $Q(s,a):[B]$를 선택합니다.

## 구현 선택 이유

- **원문에서 보존한 부분:** §2 Background, Eq. (1)
- **노트북 구현:** `bellman_targets`의 terminal-aware TD target
- **이렇게 구현한 이유:** 완성된 고수준 model을 한 줄로 호출하면
  논문의 핵심 shape·mask·loss·상태 변화가 숨겨집니다. 따라서 이 계산을
  이름이 드러나는 class와 함수로 나누고 중간 tensor를 assertion으로
  검사합니다.
- **실패를 잡는 증거:** done 상태에는 bootstrap 값이 더해지지 않는지 검사를 출력 크기만 아니라
  구현 불변식으로 사용합니다.
- **축소 선택:** 논문은 Atari 화면 4장을 convolutional Q-network에 넣어 수백만 frame을 학습한다. 여기서는 one-hot 상태 5개와 2층 MLP로 Algorithm 1의 학습 역학만 재현한다.는 기본 실행에서 생략합니다.
  이는 Windows CPU에서도 반복 실험이 가능하게 하면서 핵심 메커니즘은
  유지하기 위한 선택입니다.

## 1. Atari DQN을 LineWorld로 축소하기

논문 §2 Eq. (1)의 Bellman 관계에서 한 step target은

$$y=r+\gamma(1-\text{done})\max_{a'}Q(s',a')$$

입니다. §2 Eq. (2)는 $(y-Q(s,a;\theta))^2$을 최소화하고, §4 Algorithm 1은
상관된 연속 경험 대신 replay memory에서 무작위 전이를 뽑습니다. 여기서는 상태 0과
4가 terminal인 다섯 칸 환경을 사용합니다. 4에 도착하면 +1, 0이면 -1입니다.

**성공 기준:** 학습 뒤 greedy agent의 평균 return이 0.8보다 크고, 시작 상태에서
`Q(right) > Q(left)`여야 합니다.

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** states [B,Dₛ] → action values [B,A] → selected Q/TD target [B]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
from collections import deque
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(140)
np.random.seed(140)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
# Python environment steps stay on CPU; AI_LAB_DEVICE=cpu can avoid
# accelerator overhead for this very small neural update.


class LineWorld:
    n_states = 5
    n_actions = 2  # 0=left, 1=right

    def reset(self):
        self.state = 2
        self.steps = 0
        return self.state

    def step(self, action):
        self.steps += 1
        self.state = int(np.clip(self.state + (-1 if action == 0 else 1), 0, 4))
        done = self.state in (0, 4) or self.steps >= 8
        reward = 1.0 if self.state == 4 else (-1.0 if self.state == 0 else -0.02)
        return self.state, reward, done


def encode_states(states):
    states = torch.as_tensor(states, dtype=torch.long)
    return F.one_hot(states, num_classes=LineWorld.n_states).float().to(DEVICE)


env = LineWorld()
print(ACCELERATOR.summary())
print("LineWorld and replay storage stay on CPU; neural batches use DEVICE.")
assert env.reset() == 2
assert encode_states([0, 2, 4]).shape == (3, 5)

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** §4 Deep Reinforcement Learning, Algorithm 1
- **이 셀의 책임:** `ReplayBuffer`와 `epsilon_greedy`의 수집·표본화·탐험 경로
- **Tensor shape 계약:** states [B,Dₛ] → action values [B,A] → selected Q/TD target [B]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** buffer batch 계약과 학습 뒤 greedy return을 검사. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity):
        self.capacity = capacity
        self.storage = []
        self.position = 0

    def __len__(self):
        return len(self.storage)

    def push(self, state, action, reward, next_state, done):
        transition = (state, action, reward, next_state, done)
        if len(self.storage) < self.capacity:
            self.storage.append(transition)
        else:
            self.storage[self.position] = transition
        self.position = (self.position + 1) % self.capacity

    def sample(self, batch_size, rng):
        indices = rng.choice(len(self.storage), size=batch_size, replace=False)
        batch = [self.storage[int(index)] for index in indices]
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            torch.tensor(states, dtype=torch.long),
            torch.tensor(actions, dtype=torch.long),
            torch.tensor(rewards, dtype=torch.float32),
            torch.tensor(next_states, dtype=torch.long),
            torch.tensor(dones, dtype=torch.float32),
        )


probe_buffer = ReplayBuffer(3)
for i in range(5):
    probe_buffer.push(i % 4, i % 2, float(i), (i + 1) % 5, False)
assert len(probe_buffer) == 3
probe_batch = probe_buffer.sample(2, np.random.default_rng(141))
assert all(tensor.shape == (2,) for tensor in probe_batch)

### 구현 단계 3 · 핵심 연산·모델

- **원문 위치:** §2 Background, Eq. (1) / §2, Eq. (2)–(3)
- **이 셀의 책임:** `bellman_targets`의 terminal-aware TD target / `QNetwork`가 선택한 action의 Q와 고정 target 사이 오차를 최소화
- **Tensor shape 계약:** states [B,Dₛ] → action values [B,A] → selected Q/TD target [B]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** done 상태에는 bootstrap 값이 더해지지 않는지 검사 / Q-value shape, finite loss와 gradient를 검사. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
class QNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(LineWorld.n_states, 24),
            nn.ReLU(),
            nn.Linear(24, LineWorld.n_actions),
        )

    def forward(self, encoded_state):
        return self.net(encoded_state)


def bellman_targets(rewards, dones, next_q_values, gamma=0.95):
    # Eq. (2)의 theta_(i-1) 역할: target을 gradient graph에서 분리한다.
    best_next = next_q_values.max(dim=1).values.detach()
    return rewards + gamma * (1.0 - dones) * best_next


def hard_sync_target(online_network, target_network):
    target_network.load_state_dict(online_network.state_dict())
    target_network.requires_grad_(False)
    target_network.eval()


def parameter_distance(first_network, second_network):
    squared_distance = 0.0
    parameter_pairs = zip(
        first_network.parameters(),
        second_network.parameters(),
        strict=True,
    )
    for first_parameter, second_parameter in parameter_pairs:
        difference = first_parameter.detach() - second_parameter.detach()
        squared_distance = squared_distance + difference.square().sum()
    return float(torch.sqrt(squared_distance).cpu())


q_network = QNetwork().to(DEVICE)
target_network = QNetwork().to(DEVICE)
hard_sync_target(q_network, target_network)
probe_next_q = torch.tensor([[2.0, 3.0], [8.0, 9.0]], requires_grad=True)
probe_target = bellman_targets(
    torch.tensor([1.0, -1.0]),
    torch.tensor([0.0, 1.0]),
    probe_next_q,
)
assert torch.allclose(probe_target, torch.tensor([3.85, -1.0]))
assert not probe_target.requires_grad
assert q_network(encode_states([0, 1])).shape == (2, 2)
assert q_network is not target_network
assert all(not parameter.requires_grad for parameter in target_network.parameters())
assert parameter_distance(q_network, target_network) == 0.0

### 구현 단계 4 · 목적함수·학습 update

- **원문 위치:** §2 Background, Eq. (1) / §2, Eq. (2)–(3) / §4 Deep Reinforcement Learning, Algorithm 1
- **이 셀의 책임:** `bellman_targets`의 terminal-aware TD target / `QNetwork`가 선택한 action의 Q와 고정 target 사이 오차를 최소화 / `ReplayBuffer`와 `epsilon_greedy`의 수집·표본화·탐험 경로
- **Tensor shape 계약:** states [B,Dₛ] → action values [B,A] → selected Q/TD target [B]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** done 상태에는 bootstrap 값이 더해지지 않는지 검사 / Q-value shape, finite loss와 gradient를 검사 / buffer batch 계약과 학습 뒤 greedy return을 검사. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def epsilon_greedy(network, state, epsilon, rng):
    if rng.random() < epsilon:
        return int(rng.integers(LineWorld.n_actions))
    with torch.no_grad():
        return int(network(encode_states([state])).argmax(dim=1).item())


replay = ReplayBuffer(capacity=400)
optimizer = torch.optim.Adam(q_network.parameters(), lr=8e-3)
rng = np.random.default_rng(142)
returns, losses = [], []
sync_interval = 40
update_count = 0
pre_sync_distances = []
post_sync_distances = []

for episode in range(180):
    state = env.reset()
    episode_return = 0.0
    epsilon = max(0.05, 0.85 * (1.0 - episode / 150.0))
    done = False
    while not done:
        action = epsilon_greedy(q_network, state, epsilon, rng)
        next_state, reward, done = env.step(action)
        replay.push(state, action, reward, next_state, done)
        state = next_state
        episode_return += reward

        if len(replay) >= 32:
            states, actions, rewards, next_states, dones = replay.sample(32, rng)
            states, actions, rewards, next_states, dones = ACCELERATOR.move(
                states, actions, rewards, next_states, dones
            )
            predicted_q = q_network(encode_states(states))
            predicted_q = predicted_q.gather(
                1,
                actions[:, None],
            ).squeeze(1)
            with torch.no_grad():
                next_q = target_network(encode_states(next_states))
                targets = bellman_targets(rewards, dones, next_q)
            loss = F.mse_loss(predicted_q, targets)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            losses.append(float(loss.detach()))
            update_count += 1

            if update_count % sync_interval == 0:
                distance_before = parameter_distance(
                    q_network,
                    target_network,
                )
                pre_sync_distances.append(distance_before)
                assert distance_before > 0.0
                hard_sync_target(q_network, target_network)
                distance_after = parameter_distance(
                    q_network,
                    target_network,
                )
                post_sync_distances.append(distance_after)
                assert distance_after == 0.0

    returns.append(episode_return)

print(f"episodes={len(returns)}, replay={len(replay)}, updates={len(losses)}")
assert len(returns) == 180 and len(losses) > 100
assert np.isfinite(losses).all()
assert pre_sync_distances and min(pre_sync_distances) > 0.0
assert post_sync_distances and max(post_sync_distances) == 0.0
assert all(parameter.grad is None for parameter in target_network.parameters())

### 구현 단계 5 · 평가·완료 증거

- **원문 위치:** §4 Deep Reinforcement Learning, Algorithm 1
- **이 셀의 책임:** `ReplayBuffer`와 `epsilon_greedy`의 수집·표본화·탐험 경로
- **Tensor shape 계약:** states [B,Dₛ] → action values [B,A] → selected Q/TD target [B]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** buffer batch 계약과 학습 뒤 greedy return을 검사. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
evaluation_returns = []
eval_rng = np.random.default_rng(143)
for _ in range(40):
    state = env.reset()
    total = 0.0
    done = False
    while not done:
        action = epsilon_greedy(q_network, state, 0.0, eval_rng)
        state, reward, done = env.step(action)
        total += reward
    evaluation_returns.append(total)

with torch.no_grad():
    all_states = encode_states(torch.arange(LineWorld.n_states))
    all_q = q_network(all_states).detach().cpu().numpy()
mean_return = float(np.mean(evaluation_returns))
print(f"greedy mean return={mean_return:.3f}")
print("Q(start=2) [left, right] =", np.round(all_q[2], 3).tolist())
assert mean_return > 0.80
assert all_q[2, 1] > all_q[2, 0]

moving_return = np.convolve(returns, np.ones(15) / 15, mode="valid")
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
axes[0].plot(returns, alpha=0.3, label="episode return")
axes[0].plot(np.arange(14, len(returns)), moving_return, label="15-episode mean")
axes[0].set(title="epsilon-greedy learning", xlabel="episode")
axes[0].legend()
axes[1].plot(losses)
axes[1].set(title="replay minibatch MSE", xlabel="update")
states_axis = np.arange(LineWorld.n_states)
axes[2].plot(states_axis, all_q[:, 0], "o-", label="left")
axes[2].plot(states_axis, all_q[:, 1], "o-", label="right")
axes[2].set(title="learned action values", xlabel="state", ylabel="Q")
axes[2].legend()
fig.tight_layout()
plt.show()

## 7. 해석 예시 답안

1. 시간적으로 강하게 상관된 표본을 섞어 update 분산을 낮추고, 한 경험을 여러 번 재사용해 데이터 효율을 높입니다.
2. Eq. (2)는 이전/고정 Q가 만든 target을 상수로 보고 현재 Q만 회귀합니다.
   detach하지 않으면 target 쪽 Q도 동시에 움직여 다른 목적함수가 됩니다.
3. 원 논문은 최근 4개 Atari frame을 convolution으로 표현하지만, 여기서는
   Markov 상태를 이미 알고 one-hot 벡터로 제공합니다. replay와 Bellman
   update는 같지만 시각표현 학습은 생략했습니다.

## 마무리 해설과 다음 실험

1. 핵심 연산과 구현 위치를 원 논문의 용어로 다시 연결해 설명하세요.
2. 이 미니 실험은 `논문은 Atari 화면 4장을 convolutional Q-network에 넣어 수백만 frame을 학습한다. 여기서는 one-hot 상태 5개와 2층 MLP로 Algorithm 1의 학습 역학만 재현한다.`를 재현하지 않으므로 논문의 최종 성능을 검증한 것이 아닙니다.
3. 다음에는 데이터 크기, 모델 폭, 학습 횟수 중 **하나만** 바꾸고 같은 seed에서 metric을 비교하세요.

정답의 수치 자체보다, shape·확률 정규화·mask·gradient 같은 불변식을 먼저 확인하는 습관이
논문 구현에서 가장 강한 디버깅 도구입니다.